In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path("/content/drive/MyDrive/bluestock_mf_capstone/data/raw")
PROCESSED = Path("/content/drive/MyDrive/bluestock_mf_capstone/data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

print("Drive mounted and paths set!")

Mounted at /content/drive
Drive mounted and paths set!


In [3]:
nav = pd.read_csv(RAW / "02_nav_history.csv")
print(f"Before cleaning: {nav.shape}")

nav['date'] = pd.to_datetime(nav['date'])
nav = nav.sort_values(['amfi_code', 'date']).reset_index(drop=True)

nav_filled = []
for code, group in nav.groupby('amfi_code'):
    group = group.set_index('date')
    full_range = pd.date_range(group.index.min(), group.index.max(), freq='D')
    group = group.reindex(full_range)
    group['amfi_code'] = code
    group['nav'] = group['nav'].ffill()
    group.index.name = 'date'
    nav_filled.append(group.reset_index())

nav_clean = pd.concat(nav_filled).reset_index(drop=True)
nav_clean = nav_clean.drop_duplicates(subset=['amfi_code', 'date'])
nav_clean = nav_clean[nav_clean['nav'] > 0]

print(f"After cleaning: {nav_clean.shape}")
print(f"Nulls: {nav_clean.isnull().sum().sum()}")

nav_clean.to_csv(PROCESSED / "02_nav_history_clean.csv", index=False)
print("Saved to processed!")

Before cleaning: (46000, 3)
After cleaning: (64320, 3)
Nulls: 0
Saved to processed!


In [4]:
# Load investor_transactions
txn = pd.read_csv(RAW / "08_investor_transactions.csv")
print(f"Before cleaning: {txn.shape}")

# Fix date format
txn['transaction_date'] = pd.to_datetime(txn['transaction_date'])

# Standardise transaction_type
txn['transaction_type'] = txn['transaction_type'].str.strip().str.title()
print(f"\nTransaction types: {txn['transaction_type'].unique()}")

# Validate amount > 0
txn = txn[txn['amount_inr'] > 0]

# Check KYC status values
print(f"KYC status values: {txn['kyc_status'].unique()}")

# Remove duplicates
txn = txn.drop_duplicates()

print(f"\nAfter cleaning: {txn.shape}")
print(f"Nulls: {txn.isnull().sum().sum()}")

# Save
txn.to_csv(PROCESSED / "08_investor_transactions_clean.csv", index=False)
print("Saved to processed!")

Before cleaning: (32778, 13)

Transaction types: ['Sip' 'Redemption' 'Lumpsum']
KYC status values: ['Verified' 'Pending']

After cleaning: (32778, 13)
Nulls: 0
Saved to processed!


In [5]:
# Load scheme_performance
perf = pd.read_csv(RAW / "07_scheme_performance.csv")
print(f"Before cleaning: {perf.shape}")

# Validate return values are numeric
return_cols = ['return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct']
for col in return_cols:
    print(f"{col} — min: {perf[col].min()}, max: {perf[col].max()}")

# Check expense_ratio range (0.1% - 2.5%)
exp_anomalies = perf[(perf['expense_ratio_pct'] < 0.1) | (perf['expense_ratio_pct'] > 2.5)]
print(f"\nExpense ratio anomalies: {len(exp_anomalies)}")
if len(exp_anomalies) > 0:
    print(exp_anomalies[['scheme_name', 'expense_ratio_pct']])

# Remove duplicates
perf = perf.drop_duplicates()

print(f"\nAfter cleaning: {perf.shape}")
print(f"Nulls: {perf.isnull().sum().sum()}")

# Save
perf.to_csv(PROCESSED / "07_scheme_performance_clean.csv", index=False)
print("Saved to processed!")

Before cleaning: (40, 19)
return_1yr_pct — min: 4.26, max: 24.93
return_3yr_pct — min: 5.14, max: 23.39
return_5yr_pct — min: 5.43, max: 23.8

Expense ratio anomalies: 0

After cleaning: (40, 19)
Nulls: 0
Saved to processed!


In [6]:
# Clean and save remaining datasets
files = [
    "01_fund_master.csv",
    "03_aum_by_fund_house.csv",
    "04_monthly_sip_inflows.csv",
    "05_category_inflows.csv",
    "06_industry_folio_count.csv",
    "09_portfolio_holdings.csv",
    "10_benchmark_indices.csv"
]

for filename in files:
    df = pd.read_csv(RAW / filename)

    # Fix date columns if present
    for col in df.columns:
        if 'date' in col.lower() or 'month' in col.lower():
            df[col] = pd.to_datetime(df[col])

    # Fill yoy_growth_pct nulls with 0 in sip_inflows
    if 'yoy_growth_pct' in df.columns:
        df['yoy_growth_pct'] = df['yoy_growth_pct'].fillna(0)

    # Remove duplicates
    df = df.drop_duplicates()

    # Save to processed
    out_name = filename.replace(".csv", "_clean.csv")
    df.to_csv(PROCESSED / out_name, index=False)
    print(f"{filename} → {df.shape} → saved")

print("\nAll datasets cleaned and saved!")

01_fund_master.csv → (40, 15) → saved
03_aum_by_fund_house.csv → (90, 5) → saved
04_monthly_sip_inflows.csv → (48, 6) → saved
05_category_inflows.csv → (144, 3) → saved
06_industry_folio_count.csv → (21, 6) → saved
09_portfolio_holdings.csv → (322, 8) → saved
10_benchmark_indices.csv → (8050, 3) → saved

All datasets cleaned and saved!


In [7]:
schema_sql = """
-- Dimension Tables
CREATE TABLE IF NOT EXISTS dim_fund (
    amfi_code INTEGER PRIMARY KEY,
    fund_house TEXT,
    scheme_name TEXT,
    category TEXT,
    sub_category TEXT,
    plan TEXT,
    launch_date DATE,
    benchmark TEXT,
    expense_ratio_pct REAL,
    exit_load_pct REAL,
    min_sip_amount INTEGER,
    min_lumpsum_amount INTEGER,
    fund_manager TEXT,
    risk_category TEXT,
    sebi_category_code TEXT
);

CREATE TABLE IF NOT EXISTS dim_date (
    date DATE PRIMARY KEY,
    year INTEGER,
    month INTEGER,
    quarter INTEGER,
    day_of_week INTEGER,
    is_weekend INTEGER
);

-- Fact Tables
CREATE TABLE IF NOT EXISTS fact_nav (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    amfi_code INTEGER,
    date DATE,
    nav REAL,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);

CREATE TABLE IF NOT EXISTS fact_transactions (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    investor_id TEXT,
    transaction_date DATE,
    amfi_code INTEGER,
    transaction_type TEXT,
    amount_inr INTEGER,
    state TEXT,
    city TEXT,
    city_tier TEXT,
    age_group TEXT,
    gender TEXT,
    annual_income_lakh REAL,
    payment_mode TEXT,
    kyc_status TEXT,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);

CREATE TABLE IF NOT EXISTS fact_performance (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    amfi_code INTEGER,
    return_1yr_pct REAL,
    return_3yr_pct REAL,
    return_5yr_pct REAL,
    alpha REAL,
    beta REAL,
    sharpe_ratio REAL,
    sortino_ratio REAL,
    std_dev_ann_pct REAL,
    max_drawdown_pct REAL,
    aum_crore INTEGER,
    morningstar_rating INTEGER,
    risk_grade TEXT,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);

CREATE TABLE IF NOT EXISTS fact_aum (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    date DATE,
    fund_house TEXT,
    aum_lakh_crore REAL,
    aum_crore INTEGER,
    num_schemes INTEGER
);
"""

# Save schema.sql locally in Colab
with open("/content/schema.sql", "w") as f:
    f.write(schema_sql)

print("schema.sql created!")
print(schema_sql)

schema.sql created!

-- Dimension Tables
CREATE TABLE IF NOT EXISTS dim_fund (
    amfi_code INTEGER PRIMARY KEY,
    fund_house TEXT,
    scheme_name TEXT,
    category TEXT,
    sub_category TEXT,
    plan TEXT,
    launch_date DATE,
    benchmark TEXT,
    expense_ratio_pct REAL,
    exit_load_pct REAL,
    min_sip_amount INTEGER,
    min_lumpsum_amount INTEGER,
    fund_manager TEXT,
    risk_category TEXT,
    sebi_category_code TEXT
);

CREATE TABLE IF NOT EXISTS dim_date (
    date DATE PRIMARY KEY,
    year INTEGER,
    month INTEGER,
    quarter INTEGER,
    day_of_week INTEGER,
    is_weekend INTEGER
);

-- Fact Tables
CREATE TABLE IF NOT EXISTS fact_nav (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    amfi_code INTEGER,
    date DATE,
    nav REAL,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);

CREATE TABLE IF NOT EXISTS fact_transactions (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    investor_id TEXT,
    transaction_date DATE,
    amfi_code INTEGER,
    tra

In [8]:
import sqlite3
from sqlalchemy import create_engine

# Create SQLite DB in Colab
DB_PATH = "/content/bluestock_mf.db"
engine = create_engine(f"sqlite:///{DB_PATH}")

# Execute schema
with sqlite3.connect(DB_PATH) as conn:
    with open("/content/schema.sql") as f:
        conn.executescript(f.read())
print("Schema applied!")

# Load dim_fund
fund = pd.read_csv(PROCESSED / "01_fund_master_clean.csv")
fund.to_sql("dim_fund", engine, if_exists="replace", index=False)
print(f"dim_fund: {len(fund)} rows loaded")

# Load dim_date from nav dates
nav = pd.read_csv(PROCESSED / "02_nav_history_clean.csv", parse_dates=['date'])
dates = pd.DataFrame({'date': nav['date'].unique()})
dates['year'] = dates['date'].dt.year
dates['month'] = dates['date'].dt.month
dates['quarter'] = dates['date'].dt.quarter
dates['day_of_week'] = dates['date'].dt.dayofweek
dates['is_weekend'] = dates['day_of_week'].isin([5, 6]).astype(int)
dates.to_sql("dim_date", engine, if_exists="replace", index=False)
print(f"dim_date: {len(dates)} rows loaded")

# Load fact_nav
nav.to_sql("fact_nav", engine, if_exists="replace", index=False)
print(f"fact_nav: {len(nav)} rows loaded")

# Load fact_transactions
txn = pd.read_csv(PROCESSED / "08_investor_transactions_clean.csv")
txn.to_sql("fact_transactions", engine, if_exists="replace", index=False)
print(f"fact_transactions: {len(txn)} rows loaded")

# Load fact_performance
perf = pd.read_csv(PROCESSED / "07_scheme_performance_clean.csv")
perf.to_sql("fact_performance", engine, if_exists="replace", index=False)
print(f"fact_performance: {len(perf)} rows loaded")

# Load fact_aum
aum = pd.read_csv(PROCESSED / "03_aum_by_fund_house_clean.csv")
aum.to_sql("fact_aum", engine, if_exists="replace", index=False)
print(f"fact_aum: {len(aum)} rows loaded")

print("\nAll tables loaded into SQLite!")

Schema applied!
dim_fund: 40 rows loaded
dim_date: 1608 rows loaded
fact_nav: 64320 rows loaded
fact_transactions: 32778 rows loaded
fact_performance: 40 rows loaded
fact_aum: 90 rows loaded

All tables loaded into SQLite!


In [10]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(DB_PATH)

queries = {
    "1. Top 5 funds by AUM": """
        SELECT f.scheme_name, p.aum_crore
        FROM fact_performance p
        JOIN dim_fund f ON p.amfi_code = f.amfi_code
        ORDER BY p.aum_crore DESC
        LIMIT 5
    """,
    "2. Average NAV per month (all funds)": """
        SELECT amfi_code, strftime('%Y-%m', date) AS month, ROUND(AVG(nav), 2) AS avg_nav
        FROM fact_nav
        GROUP BY amfi_code, month
        ORDER BY amfi_code, month
        LIMIT 15
    """,
    "3. Transactions by state": """
        SELECT state, COUNT(*) AS total_txns, ROUND(SUM(amount_inr)/1e7, 2) AS total_crore
        FROM fact_transactions
        GROUP BY state
        ORDER BY total_txns DESC
    """,
    "4. Funds with expense_ratio < 1%": """
        SELECT scheme_name, fund_house, expense_ratio_pct
        FROM dim_fund
        WHERE expense_ratio_pct < 1.0
        ORDER BY expense_ratio_pct ASC
    """,
    "5. Top 5 funds by 5yr return": """
        SELECT f.scheme_name, p.return_5yr_pct
        FROM fact_performance p
        JOIN dim_fund f ON p.amfi_code = f.amfi_code
        ORDER BY p.return_5yr_pct DESC
        LIMIT 5
    """,
    "6. Transaction count and amount by type": """
        SELECT transaction_type, COUNT(*) AS count,
               ROUND(SUM(amount_inr)/1e7, 2) AS total_crore
        FROM fact_transactions
        GROUP BY transaction_type
        ORDER BY count DESC
    """,
    "7. Average Sharpe ratio by category": """
        SELECT f.category, ROUND(AVG(p.sharpe_ratio), 3) AS avg_sharpe
        FROM fact_performance p
        JOIN dim_fund f ON p.amfi_code = f.amfi_code
        GROUP BY f.category
    """,
    "8. Top 5 funds by alpha": """
        SELECT f.scheme_name, p.alpha, p.beta, p.sharpe_ratio
        FROM fact_performance p
        JOIN dim_fund f ON p.amfi_code = f.amfi_code
        ORDER BY p.alpha DESC
        LIMIT 5
    """,
    "9. Latest AUM by fund house": """
        SELECT fund_house, aum_crore
        FROM fact_aum
        WHERE date = (SELECT MAX(date) FROM fact_aum)
        ORDER BY aum_crore DESC
    """,
    "10. Transactions by gender and city tier": """
        SELECT gender, city_tier, COUNT(*) AS count,
               ROUND(AVG(amount_inr), 0) AS avg_amount
        FROM fact_transactions
        GROUP BY gender, city_tier
        ORDER BY gender, city_tier
    """
}

for title, query in queries.items():
    print(f"\n{'='*55}")
    print(f"Query {title}")
    print('='*55)
    try:
        result = pd.read_sql_query(query, conn)
        print(result.to_string(index=False))
    except Exception as e:
        print(f"ERROR: {e}")

conn.close()
print("\nAll 10 queries done!")


Query 1. Top 5 funds by AUM
                                          scheme_name  aum_crore
Mirae Asset Emerging Bluechip Fund - Regular - Growth      49046
        Kotak Emerging Equity Fund - Regular - Growth      47469
       Nippon India Small Cap Fund - Regular - Growth      43630
           DSP Top 100 Equity Fund - Regular - Growth      41828
                  UTI Mid Cap Fund - Regular - Growth      41728

Query 2. Average NAV per month (all funds)
 amfi_code   month  avg_nav
    100016 2022-01   511.92
    100016 2022-02   514.54
    100016 2022-03   522.29
    100016 2022-04   526.11
    100016 2022-05   504.34
    100016 2022-06   465.38
    100016 2022-07   436.77
    100016 2022-08   420.92
    100016 2022-09   422.43
    100016 2022-10   431.35
    100016 2022-11   463.84
    100016 2022-12   480.87
    100016 2023-01   490.85
    100016 2023-02   492.45
    100016 2023-03   545.74

Query 3. Transactions by state
         state  total_txns  total_crore
        Punjab   

In [11]:
queries_sql = """-- Bluestock MF Capstone — 10 Analytical Queries

-- Q1. Top 5 funds by AUM
SELECT f.scheme_name, p.aum_crore
FROM fact_performance p
JOIN dim_fund f ON p.amfi_code = f.amfi_code
ORDER BY p.aum_crore DESC
LIMIT 5;

-- Q2. Average NAV per month per fund
SELECT amfi_code, strftime('%Y-%m', date) AS month, ROUND(AVG(nav), 2) AS avg_nav
FROM fact_nav
GROUP BY amfi_code, month
ORDER BY amfi_code, month;

-- Q3. Transactions by state
SELECT state, COUNT(*) AS total_txns, ROUND(SUM(amount_inr)/1e7, 2) AS total_crore
FROM fact_transactions
GROUP BY state
ORDER BY total_txns DESC;

-- Q4. Funds with expense_ratio < 1%
SELECT scheme_name, fund_house, expense_ratio_pct
FROM dim_fund
WHERE expense_ratio_pct < 1.0
ORDER BY expense_ratio_pct ASC;

-- Q5. Top 5 funds by 5yr return
SELECT f.scheme_name, p.return_5yr_pct
FROM fact_performance p
JOIN dim_fund f ON p.amfi_code = f.amfi_code
ORDER BY p.return_5yr_pct DESC
LIMIT 5;

-- Q6. Transaction count and amount by type
SELECT transaction_type, COUNT(*) AS count,
       ROUND(SUM(amount_inr)/1e7, 2) AS total_crore
FROM fact_transactions
GROUP BY transaction_type
ORDER BY count DESC;

-- Q7. Average Sharpe ratio by category
SELECT f.category, ROUND(AVG(p.sharpe_ratio), 3) AS avg_sharpe
FROM fact_performance p
JOIN dim_fund f ON p.amfi_code = f.amfi_code
GROUP BY f.category;

-- Q8. Top 5 funds by alpha
SELECT f.scheme_name, p.alpha, p.beta, p.sharpe_ratio
FROM fact_performance p
JOIN dim_fund f ON p.amfi_code = f.amfi_code
ORDER BY p.alpha DESC
LIMIT 5;

-- Q9. Latest AUM by fund house
SELECT fund_house, aum_crore
FROM fact_aum
WHERE date = (SELECT MAX(date) FROM fact_aum)
ORDER BY aum_crore DESC;

-- Q10. Transactions by gender and city tier
SELECT gender, city_tier, COUNT(*) AS count,
       ROUND(AVG(amount_inr), 0) AS avg_amount
FROM fact_transactions
GROUP BY gender, city_tier
ORDER BY gender, city_tier;
"""

with open("/content/queries.sql", "w") as f:
    f.write(queries_sql)

print("queries.sql saved!")

queries.sql saved!


In [12]:
data_dict = """# Data Dictionary — Bluestock MF Capstone

## 1. dim_fund (source: 01_fund_master.csv)
| Column | Type | Description |
|--------|------|-------------|
| amfi_code | INTEGER | Primary key. Unique AMFI scheme code assigned by SEBI |
| fund_house | TEXT | Name of the Asset Management Company (AMC) |
| scheme_name | TEXT | Full name of the mutual fund scheme |
| category | TEXT | Broad category — Equity or Debt |
| sub_category | TEXT | SEBI sub-category e.g. Large Cap, Mid Cap, Gilt |
| plan | TEXT | Direct or Regular plan |
| launch_date | DATE | Date the scheme was launched |
| benchmark | TEXT | Index used to benchmark fund performance |
| expense_ratio_pct | REAL | Annual fee charged by AMC as % of AUM (range: 0.1–2.5%) |
| exit_load_pct | REAL | Fee charged on redemption before lock-in period |
| min_sip_amount | INTEGER | Minimum SIP investment amount in INR |
| min_lumpsum_amount | INTEGER | Minimum lump sum investment amount in INR |
| fund_manager | TEXT | Name of the fund manager |
| risk_category | TEXT | SEBI risk grade: Low / Moderate / Moderately High / High / Very High |
| sebi_category_code | TEXT | SEBI internal category code |

## 2. dim_date (derived from fact_nav dates)
| Column | Type | Description |
|--------|------|-------------|
| date | DATE | Primary key. Calendar date |
| year | INTEGER | Calendar year |
| month | INTEGER | Month number (1–12) |
| quarter | INTEGER | Quarter number (1–4) |
| day_of_week | INTEGER | Day of week (0=Monday, 6=Sunday) |
| is_weekend | INTEGER | 1 if Saturday or Sunday, else 0 |

## 3. fact_nav (source: 02_nav_history.csv)
| Column | Type | Description |
|--------|------|-------------|
| amfi_code | INTEGER | Foreign key → dim_fund |
| date | DATE | NAV date. Forward-filled for weekends and holidays |
| nav | REAL | Net Asset Value in INR per unit. Must be > 0 |

## 4. fact_transactions (source: 08_investor_transactions.csv)
| Column | Type | Description |
|--------|------|-------------|
| investor_id | TEXT | Unique investor identifier |
| transaction_date | DATE | Date of the transaction |
| amfi_code | INTEGER | Foreign key → dim_fund |
| transaction_type | TEXT | Standardised: SIP / Lumpsum / Redemption |
| amount_inr | INTEGER | Transaction amount in INR. Must be > 0 |
| state | TEXT | Indian state of the investor |
| city | TEXT | City of the investor |
| city_tier | TEXT | Tier 1 / Tier 2 / Tier 3 |
| age_group | TEXT | Age bracket of the investor |
| gender | TEXT | Gender of the investor |
| annual_income_lakh | REAL | Annual income in lakhs INR |
| payment_mode | TEXT | Payment method e.g. UPI, Net Banking |
| kyc_status | TEXT | KYC verification status: Verified or Pending |

## 5. fact_performance (source: 07_scheme_performance.csv)
| Column | Type | Description |
|--------|------|-------------|
| amfi_code | INTEGER | Foreign key → dim_fund |
| return_1yr_pct | REAL | Trailing 1-year return in % |
| return_3yr_pct | REAL | Trailing 3-year CAGR in % |
| return_5yr_pct | REAL | Trailing 5-year CAGR in % |
| alpha | REAL | Excess return over benchmark |
| beta | REAL | Sensitivity to market movements |
| sharpe_ratio | REAL | Risk-adjusted return (return per unit of risk) |
| sortino_ratio | REAL | Downside risk-adjusted return |
| std_dev_ann_pct | REAL | Annualised standard deviation of returns in % |
| max_drawdown_pct | REAL | Maximum peak-to-trough decline in % |
| aum_crore | INTEGER | Assets Under Management in crore INR |
| morningstar_rating | INTEGER | Morningstar star rating (1–5) |
| risk_grade | TEXT | Internal risk classification |

## 6. fact_aum (source: 03_aum_by_fund_house.csv)
| Column | Type | Description |
|--------|------|-------------|
| date | DATE | Reporting date |
| fund_house | TEXT | Name of the AMC |
| aum_lakh_crore | REAL | Total AUM in lakh crore INR |
| aum_crore | INTEGER | Total AUM in crore INR |
| num_schemes | INTEGER | Number of schemes managed by the fund house |
"""

with open("/content/data_dictionary.md", "w") as f:
    f.write(data_dict)

print("data_dictionary.md saved!")

data_dictionary.md saved!


In [13]:
import shutil

DRIVE_SQL = Path("/content/drive/MyDrive/bluestock_mf_capstone/sql")
DRIVE_SQL.mkdir(parents=True, exist_ok=True)

DRIVE_REPORTS = Path("/content/drive/MyDrive/bluestock_mf_capstone/reports")
DRIVE_REPORTS.mkdir(parents=True, exist_ok=True)

DRIVE_DB = Path("/content/drive/MyDrive/bluestock_mf_capstone/data/db")
DRIVE_DB.mkdir(parents=True, exist_ok=True)

# Copy files to Drive
shutil.copy("/content/schema.sql", DRIVE_SQL / "schema.sql")
shutil.copy("/content/queries.sql", DRIVE_SQL / "queries.sql")
shutil.copy("/content/data_dictionary.md", DRIVE_REPORTS / "data_dictionary.md")
shutil.copy("/content/bluestock_mf.db", DRIVE_DB / "bluestock_mf.db")

print("All deliverables copied to Drive!")
print(f"  sql/schema.sql")
print(f"  sql/queries.sql")
print(f"  reports/data_dictionary.md")
print(f"  data/db/bluestock_mf.db")

All deliverables copied to Drive!
  sql/schema.sql
  sql/queries.sql
  reports/data_dictionary.md
  data/db/bluestock_mf.db
